# Object Detection mAP

**Difficulty**: 🟡 Medium

### Problem Statement

**Mean Average Precision (mAP)** is the headline number for object detection (PASCAL VOC, COCO, LVIS), instance segmentation, and open-vocabulary detection and grounding. Small differences in evaluation code are a classic source of irreproducible results: how the precision–recall curve is interpolated, how predictions are matched to ground truth, and whether the IoU threshold is inclusive can each move AP by points. Applied-CV interviews therefore often ask you to write mAP from scratch and to name these conventions.

You will implement:
1. the all-point interpolated **Average Precision** of a precision–recall curve, and
2. **mAP@IoU** for a detector's output over a set of images.

---

### Requirements

1. **`average_precision(recall, precision) -> float`** — `recall` and `precision` are 1-D tensors of equal length $K$. They are the PR points of a ranked list after each prediction, so `recall` is non-decreasing. Compute the **all-point interpolated AP** (PASCAL VOC ≥ 2010, devkit `voc_ap(..., use_07_metric=False)`):
   - add **sentinels**: recall $0$ at the start and recall $1$ at the end, both with precision $0$;
   - build the **precision envelope**, monotonically non-increasing from the right: $\hat p_i = \max_{j \ge i} p_j$;
   - sum the area of the step function over the indices where recall changes: $\mathrm{AP} = \sum_i (r_{i+1} - r_i)\,\hat p_{i+1}$.

   Equivalently $\mathrm{AP} = \sum_{k=1}^{K} (r_k - r_{k-1}) \max_{j \ge k} p_j$ with $r_0 = 0$. Recall above the last point is never reached and contributes 0, because the sentinel precision there is 0. Empty input → `0.0`.

2. **`mean_average_precision(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_threshold=0.5) -> (float, dict)`** — every argument is a **list with one entry per image**: `pred_boxes[i]` `(P_i, 4)`, `pred_scores[i]` `(P_i,)`, `pred_labels[i]` `(P_i,)` int64, `gt_boxes[i]` `(G_i, 4)`, `gt_labels[i]` `(G_i,)` int64. Boxes are `(x1, y1, x2, y2)`. Any image may have $P_i = 0$ or $G_i = 0$. For every class $c$ that has at least one GT box:
   1. Gather the class-$c$ predictions from **all images** and sort them by score, highest first.
   2. Walk them in that order. Look at the class-$c$ GT boxes **in the same image** that are **not yet matched**, and pick the one with the **highest IoU**. If that IoU is **≥ `iou_threshold`**, the prediction is a TP and that GT becomes matched. Otherwise it is an FP. Each GT is matched at most once, so a second detection of the same object is an FP.
   3. Cumulative sums give $\text{precision}_k = TP_k / k$ and $\text{recall}_k = TP_k / N^{GT}_c$, where $N^{GT}_c$ is the number of class-$c$ GT boxes over all images.
   4. $AP_c$ = `average_precision(recall, precision)`.

   Return `(mAP, per_class)`: `per_class` maps each class id (Python `int`) with ≥ 1 GT to its AP, and `mAP` is the mean of those values (a Python `float`). Classes that appear **only in predictions are ignored**, both in the dict and in the mean. A class with GT but no predictions has AP 0 and does count. No GT at all → `(0.0, {})`. The result must not depend on the order of the images or of the boxes inside an image. You may assume all scores are distinct.

> **Conventions.** This is VOC-style all-point AP at a single IoU threshold with **COCO-style matching**: the best *unmatched* GT, and IoU ≥ t. The original VOC devkit takes the max-IoU GT even if it is already matched (then FP) and uses IoU > t. COCO's headline metric additionally averages a 101-point interpolated AP over IoU thresholds 0.50:0.05:0.95. Those variants are in the follow-ups.

---

### Constraints

- ✅ `box_iou` from the IoU/NMS problem is provided in the setup cell — use it.
- ✅ Python loops over classes and over the sorted predictions are fine (greedy matching is sequential); use tensor ops for IoU, sorting and cumulative sums.
- ❌ No `torchmetrics`, `pycocotools` or other evaluation libraries.

---

<details>
  <summary>💡 Hint</summary>

  - Treat it as two separate problems: (1) turning a ranked list of TP/FP decisions into precision and recall arrays, and (2) turning those arrays into one number.
  - For the envelope, what does a running maximum computed from the *right* look like in PyTorch? (`torch.cummax` runs left to right…)
  - Matching only ever compares a prediction with GT boxes of the same class in the same image, but the *ranking* is global across images. Why does that matter for the precision–recall curve?
  - Keep a boolean "already matched" flag for every GT box.

</details>

---

In [ ]:
import torch

In [ ]:
def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """(N, 4) x (M, 4) -> (N, M) IoU of (x1, y1, x2, y2) boxes; 0 where the union is 0.
    (You implemented this in the IoU/NMS problem; it is given here.)"""
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    top_left = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    bottom_right = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (bottom_right - top_left).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    union = area1[:, None] + area2[None, :] - inter
    return inter / torch.where(union > 0, union, torch.ones_like(union))


# A worked example: 2 images, classes 0 = cat, 1 = dog, 2 = bird (bird has no GT).
gt_boxes = [torch.tensor([[0., 0., 10., 10.], [20., 20., 30., 30.]]),
            torch.tensor([[0., 0., 10., 10.], [50., 50., 60., 60.], [70., 70., 80., 80.]])]
gt_labels = [torch.tensor([0, 1]), torch.tensor([0, 0, 1])]
pred_boxes = [torch.tensor([[0., 0., 10., 10.],     # cat, TP
                            [1., 1., 10., 10.],     # cat, duplicate of the same cat -> FP
                            [20., 20., 30., 30.],   # dog, TP
                            [60., 60., 70., 70.],   # dog, nothing there -> FP
                            [0., 0., 10., 10.]]),   # bird: no bird GT anywhere -> ignored
              torch.tensor([[30., 30., 40., 40.],   # cat, nothing there -> FP
                            [50., 50., 60., 60.],   # cat, TP
                            [70., 70., 80., 80.]])] # dog, TP
pred_scores = [torch.tensor([0.90, 0.80, 0.70, 0.95, 0.99]), torch.tensor([0.85, 0.60, 0.50])]
pred_labels = [torch.tensor([0, 0, 1, 1, 2]), torch.tensor([0, 0, 1])]


def toy_detector(n_images=20, n_classes=3, seed=0):
    """Random GT boxes plus a noisy detector: jittered hits, duplicates, misses, background."""
    g = torch.Generator().manual_seed(seed)
    P, S, L, G, GL = [], [], [], [], []
    for _ in range(n_images):
        n = int(torch.randint(0, 6, (1,), generator=g))
        xy = torch.rand(n, 2, generator=g) * 200
        gt = torch.cat([xy, xy + 10 + torch.rand(n, 2, generator=g) * 40], dim=1)
        gl = torch.randint(0, n_classes, (n,), generator=g)
        hit = torch.rand(n, generator=g) < 0.8                       # 20% of objects are missed
        dup = torch.rand(n, generator=g) < 0.3                       # 30% detected twice
        src = torch.cat([torch.arange(n)[hit], torch.arange(n)[dup]])
        boxes = gt[src] + torch.randn(len(src), 4, generator=g) * 3
        boxes = torch.cat([torch.minimum(boxes[:, :2], boxes[:, 2:]), torch.maximum(boxes[:, :2], boxes[:, 2:])], 1)
        bg_xy = torch.rand(2, 2, generator=g) * 200                  # two background boxes
        boxes = torch.cat([boxes, torch.cat([bg_xy, bg_xy + 20], 1)])
        labels = torch.cat([gl[src], torch.randint(0, n_classes, (2,), generator=g)])
        P.append(boxes); L.append(labels); S.append(torch.rand(len(boxes), generator=g))
        G.append(gt); GL.append(gl)
    return P, S, L, G, GL

In [ ]:
def average_precision(recall: torch.Tensor, precision: torch.Tensor) -> float:
    """
    All-point interpolated AP (PASCAL VOC >= 2010).

    Args:
        recall: (K,) non-decreasing recall after each ranked prediction.
        precision: (K,) precision after each ranked prediction.

    Returns:
        AP as a Python float (0.0 for empty input).
    """
    # TODO: add sentinels (recall 0 and 1, precision 0 at both ends)
    # TODO: precision envelope p_hat[i] = max_{j >= i} p[j]
    # TODO: sum (r[i+1] - r[i]) * p_hat[i+1] where recall changes
    raise NotImplementedError

In [ ]:
# 4 GT; ranked decisions TP, FP, TP, TP -> recall .25 .25 .5 .75, precision 1 .5 .667 .75
rec = torch.tensor([0.25, 0.25, 0.50, 0.75])
prec = torch.tensor([1.0, 0.5, 2 / 3, 0.75])
ap = average_precision(rec, prec)
print(f"AP = {ap:.4f}")
assert abs(ap - 0.625) < 1e-6, "expected .25*1 + .25*.75 + .25*.75 = 0.625 (envelope lifts 2/3 to .75)"
assert abs(average_precision(torch.tensor([0.5]), torch.tensor([1.0])) - 0.5) < 1e-6, "unreached recall adds 0"
assert abs(average_precision(torch.tensor([0.5, 1.0]), torch.tensor([1.0, 1.0])) - 1.0) < 1e-6
assert average_precision(torch.zeros(0), torch.zeros(0)) == 0.0
print("average_precision tests passed")

In [ ]:
def mean_average_precision(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels,
                           iou_threshold: float = 0.5):
    """
    mAP at a single IoU threshold.

    Args:
        pred_boxes:  list over images of (P_i, 4) predicted boxes (x1, y1, x2, y2).
        pred_scores: list over images of (P_i,) confidence scores.
        pred_labels: list over images of (P_i,) int64 class ids.
        gt_boxes:    list over images of (G_i, 4) ground-truth boxes.
        gt_labels:   list over images of (G_i,) int64 class ids.
        iou_threshold: a prediction matches a GT when IoU >= this.

    Returns:
        (mAP, per_class): mAP as a float, per_class as {class_id: AP} for every class
        with at least one GT box.
    """
    # TODO: classes = ids that occur in gt_labels
    # TODO: for each class: gather predictions across images, sort by score (descending)
    # TODO: greedy matching to the best unmatched same-class GT in the same image (IoU >= thr)
    # TODO: cumulative TP/FP -> precision, recall -> average_precision
    raise NotImplementedError

In [ ]:
m_ap, per_class = mean_average_precision(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels)
rounded = {k: round(v, 4) for k, v in per_class.items()}
print(f"mAP@0.5 = {m_ap:.4f}  per class: {rounded}")
# cat: TP FP FP TP over 3 GT -> recall 1/3 1/3 1/3 2/3, precision 1 1/2 1/3 1/2 -> AP 1/3 + 1/3 * 1/2 = 0.5
# dog: FP TP TP over 2 GT   -> recall 0 1/2 1,        precision 0 1/2 2/3     -> AP 2/3 (envelope)
assert sorted(per_class) == [0, 1], "bird has no GT -> not in the dict"
assert abs(per_class[0] - 0.5) < 1e-6 and abs(per_class[1] - 2 / 3) < 1e-6
assert abs(m_ap - 7 / 12) < 1e-6

# Perfect predictions -> 1.0
P, S, L, G, GL = toy_detector(seed=1)
assert abs(mean_average_precision(G, [torch.rand(len(g)) for g in G], GL, G, GL)[0] - 1.0) < 1e-6

# A noisy detector: AP falls as the IoU threshold gets stricter
P, S, L, G, GL = toy_detector(seed=0)
for t in (0.5, 0.75, 0.9):
    print(f"toy detector mAP@{t}: {mean_average_precision(P, S, L, G, GL, iou_threshold=t)[0]:.4f}")

# Order of images and of boxes inside an image must not matter
perm = [torch.randperm(len(s)) for s in S]
shuffled = mean_average_precision([p[q] for p, q in zip(P, perm)][::-1], [s[q] for s, q in zip(S, perm)][::-1],
                                  [l[q] for l, q in zip(L, perm)][::-1], G[::-1], GL[::-1])
assert abs(shuffled[0] - mean_average_precision(P, S, L, G, GL)[0]) < 1e-9
print("mean_average_precision tests passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("detection-map", average_precision=average_precision,
      mean_average_precision=mean_average_precision)